In [1]:
import torch
import numpy as np

torch.manual_seed(42)

X = torch.randn(8, 4)
gamma = torch.ones(4)
beta = torch.zeros(4)

print("Input shape:", X.shape)

Input shape: torch.Size([8, 4])


In [2]:
def batch_norm(X, gamma, beta, eps=1e-5):

    mean = X.mean(dim=0, keepdim=True)
    var = X.var(dim=0, unbiased=False, keepdim=True)

    X_norm = (X - mean) / torch.sqrt(var + eps)

    return gamma * X_norm + beta, mean, var

In [3]:
def layer_norm(X, gamma, beta, eps=1e-5):

    mean = X.mean(dim=1, keepdim=True)
    var = X.var(dim=1, unbiased=False, keepdim=True)

    X_norm = (X - mean) / torch.sqrt(var + eps)

    return gamma * X_norm + beta

In [4]:
BN, mean, var = batch_norm(X, gamma, beta)
LN = layer_norm(X, gamma, beta)

print("BatchNorm output:\n", BN)
print("\nLayerNorm output:\n", LN)

BatchNorm output:
 tensor([[ 1.4514,  1.0362,  1.4297, -1.5293],
        [ 0.2784, -1.6824, -0.1468, -1.1647],
        [-1.0657,  1.1974, -0.7305, -1.0184],
        [-1.0429, -1.0081, -1.3592,  0.5583],
        [ 1.1840, -0.6087, -0.9058,  0.3233],
        [-1.0713,  0.6277,  1.2628,  1.2267],
        [ 0.8428,  0.8455,  0.9449,  0.9749],
        [-0.5767, -0.4076, -0.4951,  0.6292]])

LayerNorm output:
 tensor([[ 0.8716,  0.5928,  0.2209, -1.6853],
        [ 1.3440, -0.7473,  0.5552, -1.1519],
        [-0.4622,  1.6423, -0.1469, -1.0332],
        [-0.6401, -0.3735, -0.7050,  1.7186],
        [ 1.5784, -0.6330, -1.0476,  0.1023],
        [-1.6203,  0.4198,  0.1115,  1.0889],
        [ 0.4952,  0.5527, -1.7281,  0.6801],
        [-0.7452, -0.1393, -0.7894,  1.6739]])


In [5]:
def batch_norm_backward(dout, X, mean, var, gamma, eps=1e-5):

    m = X.shape[0]

    X_norm = (X - mean) / torch.sqrt(var + eps)

    dgamma = torch.sum(dout * X_norm, dim=0)
    dbeta = torch.sum(dout, dim=0)

    dX = (gamma / (m * torch.sqrt(var + eps))) * (
        m*dout
        - torch.sum(dout, dim=0, keepdim=True)
        - X_norm * torch.sum(dout * X_norm, dim=0, keepdim=True)
    )

    return dX, dgamma, dbeta

In [6]:
dout = torch.ones_like(X)

dX, dgamma, dbeta = batch_norm_backward(
    dout, X, mean, var, gamma
)

print("dX shape:", dX.shape)
print("dgamma:", dgamma)
print("dbeta:", dbeta)

dX shape: torch.Size([8, 4])
dgamma: tensor([ 5.9605e-08, -1.1921e-07,  5.9605e-08,  1.1921e-07])
dbeta: tensor([8., 8., 8., 8.])


In [7]:
print("BatchNorm mean:",
      BN.mean(dim=0))

print("BatchNorm variance:",
      BN.var(dim=0, unbiased=False))

print("\nNormalization completed successfully.")

BatchNorm mean: tensor([ 7.4506e-09, -1.4901e-08,  7.4506e-09,  1.4901e-08])
BatchNorm variance: tensor([1.0000, 1.0000, 1.0000, 1.0000])

Normalization completed successfully.
